In [12]:
import os
os.chdir('/orcd/data/omarabu/001/opitcho/assignment1-basics/')

In [13]:
import torch
from omegaconf import OmegaConf

from baseformer.nn.transformer import TransformerLM
from baseformer.nn.position import RotaryPositionalEmbedding
from baseformer.tokenization.bpe import BPETokenizer

## Configuration

Set the paths to your checkpoint and optionally a config file.
- `CHECKPOINT_PATH`: Path to the `.pt` checkpoint file
- `CONFIG_PATH`: Optional path to a `.yaml` config file. If `None`, uses the config stored in the checkpoint.

In [14]:
# ===== EDIT THESE PATHS =====
CHECKPOINT_PATH = "/orcd/data/omarabu/001/opitcho/assignment1-basics/baseformer/outputs/small_lr_1e-2/checkpoints/step_30000.pt"
CONFIG_PATH = None  # Set to a .yaml path to override checkpoint config, e.g. "baseformer/outputs/new_exp/.hydra/config.yaml"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## Load Checkpoint and Config

In [15]:
# Load checkpoint
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

print(f"Checkpoint loaded from step {checkpoint['step']}")
print(f"Tokens processed: {checkpoint.get('tokens_processed', 'N/A'):,}")

# Load config from checkpoint or external file
if CONFIG_PATH is not None:
    cfg = OmegaConf.load(CONFIG_PATH)
    print(f"Config loaded from: {CONFIG_PATH}")
else:
    cfg = OmegaConf.create(checkpoint['config'])
    print("Config loaded from checkpoint")

print("\n--- Model Config ---")
print(OmegaConf.to_yaml(cfg.model))

print("--- Position Config ---")
print(OmegaConf.to_yaml(cfg.position))

Checkpoint loaded from step 30000
Tokens processed: 491,520,000
Config loaded from checkpoint

--- Model Config ---
vocab_size: 10048
d_model: 512
num_layers: 4
num_heads: 16
d_ff: 1344

--- Position Config ---
theta: 10000.0
max_seq_len: 256



## Initialize Model

In [21]:
# Build RoPE
d_k = cfg.model.d_model // cfg.model.num_heads
print(f"Building RoPE with theta={cfg.position.theta}, d_k={d_k}, max_seq_len={cfg.position.max_seq_len}")

rope = RotaryPositionalEmbedding(
    theta=cfg.position.theta,
    d_k=d_k,
    max_seq_len=cfg.position.max_seq_len,
    device=DEVICE,
)

# Build model
model = TransformerLM(
    vocab_size=cfg.model.vocab_size,
    d_model=cfg.model.d_model,
    num_layers=cfg.model.num_layers,
    num_heads=cfg.model.num_heads,
    d_ff=cfg.model.d_ff,
    rope=rope,
    device=DEVICE,
)

# Load weights
model.load_state_dict(checkpoint['model'])
model = model.to(DEVICE)
model.eval()

num_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded with {num_params:,} parameters")

Building RoPE with theta=10000.0, d_k=32, max_seq_len=256
Model loaded with 10,289,664 parameters


## Load Tokenizer

In [22]:
tokenizer = BPETokenizer.from_files(
    cfg.data.tokenizer.vocab_path,
    cfg.data.tokenizer.merges_path,
)

print(f"Tokenizer loaded with {len(tokenizer.vocab)} tokens")
print(f"Special tokens: {tokenizer.special_tokens}")

Tokenizer loaded with 10048 tokens
Special tokens: {'<|endoftext|>'}


## Generation Function

In [23]:
@torch.no_grad()
def generate(
    prompt: str,
    max_tokens: int = 100,
    temperature: float = 1.0,
    top_p: float | None = None,
    stop_at_eos: bool = True,
    print_tokens: bool = False,
) -> str:
    """
    Generate text continuation from a prompt.
    
    Args:
        prompt: Text to continue from.
        max_tokens: Maximum number of tokens to generate.
        temperature: Sampling temperature (0 = greedy, higher = more random).
        top_p: Nucleus sampling threshold. If set, only tokens within cumulative probability top_p are considered.
        stop_at_eos: Stop generation when <|endoftext|> is generated.
        print_tokens: If True, print each token as it's generated.
    
    Returns:
        Generated text (including the prompt).
    """
    # Encode prompt
    prompt_ids = tokenizer.encode(prompt)
    token_ids = torch.tensor([prompt_ids], dtype=torch.long, device=DEVICE)
    
    eos_token = tokenizer.encode("<|endoftext|>")[0]
    
    # Generate tokens
    generated_ids = list(prompt_ids)
    for i, next_token_tensor in enumerate(model.decode(token_ids, temperature=temperature, top_p=top_p)):
        next_token = next_token_tensor.item()  # Convert tensor to int (batch=1)
        generated_ids.append(next_token)
        
        if print_tokens:
            token_str = tokenizer.decode([next_token])
            print(repr(token_str), end=" ", flush=True)
        
        if stop_at_eos and next_token == eos_token:
            break
        if i >= max_tokens - 1:
            break
    
    if print_tokens:
        print()  # Newline after token stream
    
    return tokenizer.decode(generated_ids)

In [ ]:
prompt = "Your string here"
generate()